# primesense 🎬
## Notebook 01 — Data Loading, Cleaning & Preparation

---

### 🗺️ About This Notebook

This is the first notebook in the **primesense** pipeline. Before we can train any model or draw any insight, we need clean, well-structured data. This notebook is responsible for exactly that — taking the raw Amazon Prime Video review data and transforming it into a reliable, analysis-ready dataset.

Think of this notebook as the **foundation** of the entire project. Every other notebook — EDA, modeling, BERT — depends on the output produced here. If the data going in is messy, everything downstream will be too.

---

### 🎯 What This Notebook Does

1. **Loads** the raw Amazon Prime Video reviews dataset (`prime_videos.csv`)
2. **Inspects** the structure, shape, data types and missing values
3. **Cleans** the data — handles nulls, duplicates, and irrelevant columns
4. **Engineers** new features — combines text fields, assigns sentiment labels from star ratings
5. **Validates** the final dataset is ready for EDA and modeling
6. **Saves** the cleaned dataset to `data/processed/cleaned_reviews.csv`

---

### 📦 Dataset Background

The raw data was sourced from the [Amazon Reviews 2023 dataset](https://amazon-reviews-2023.github.io/) compiled by **McAuley Lab**. It is one of the largest publicly available e-commerce review datasets, covering millions of reviews across hundreds of product categories.

For this project, we focused exclusively on the **Movies & TV** category, and further filtered it down to reviews of **Amazon Prime Video streaming content only** — excluding physical media like DVDs and Blu-Rays.

The raw dataset is a **merge of two sources:**
- `User Reviews` — ratings, review text, timestamps, verified purchase status
- `Item Metadata` — movie/show titles, genres, descriptions, prices

These were merged on `parent_asin` (the product ID), resulting in **233,610 rows** before cleaning.

---

### 🏷️ Sentiment Labelling Strategy

Since this is a supervised classification problem, we need labels. Rather than manually labelling reviews, we derive sentiment from the star rating — a standard and well-validated approach in NLP research:

| Star Rating | Sentiment Label |
|---|---|
| ⭐⭐⭐⭐⭐ (4–5) | `positive` |
| ⭐⭐⭐ (3) | `neutral` |
| ⭐⭐ (1–2) | `negative` |

This gives us a **3-class classification problem**, which is more nuanced and challenging than simple binary positive/negative sentiment.

---
## 1.0 Imports & Configuration

In [1]:
import sys
import os

# Add project root to path so we can import from src/
sys.path.insert(0, os.path.abspath('..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from src.preprocess import assign_sentiment

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
%matplotlib inline

# Load central config
with open('/mnt/d/DS_PROJECTS/primesense/config.yaml', 'r') as f:
    CONFIG = yaml.safe_load(f)

print('✅ Libraries loaded successfully.')
print(f"📁 Project: {CONFIG['project']['name']} v{CONFIG['project']['version']}")

✅ Libraries loaded successfully.
📁 Project: primesense v1.0.0


---
## 1.1 Load the Raw Dataset

We load the pre-merged CSV from `data/raw/`. This file was created by sampling 500,000 rows from each of the McAuley Lab JSON datasets and merging them on `parent_asin`. It contains **233,610 rows** representing Prime Video reviews with their associated metadata.

In [2]:
RAW_PATH = f"../{CONFIG['data']['raw_reviews']}"

df = pd.read_csv(RAW_PATH)

print(f'✅ Dataset loaded successfully.')
print(f'   Shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

✅ Dataset loaded successfully.
   Shape : 233,610 rows × 14 columns
   Memory: 195.5 MB


In [4]:
# First look at the raw data
df.head(10)

,rating,title_x,text,asin,parent_asin,timestamp,verified_purchase,main_category,title_y,average_rating,rating_number,description,price,categories
0,5,Five Stars,"Amazon, please buy the show! I'm hooked!",B013488XFS,B013488XFS,2015-08-24 03:07:17.000,True,Prime Video,Sneaky Pete,4.6,56658.0,['A\xa0con man (Giovanni Ribisi) on the run fr...,NaN,Suspense
1,5,Five Stars,My Kiddos LOVE this show!!,B00CB6VTDS,B00CB6VTDS,2016-04-19 21:16:50.000,True,Prime Video,Creative Galaxy,4.8,6403.0,['Follow the adventures of Arty and his sideki...,NaN,Kids
2,5,What Love Is...,"...isn't always how you expect it to be, but w...",B001H1SVZC,B001H1SVZC,2020-05-28 04:13:47.074,True,Prime Video,NaN,4.5,389.0,NaN,NaN,NaN
3,5,QUIRKY TURNS TO HEARTSTRINGS,As you learn about the very unique characters ...,B06WVW16WY,B06WVW16WY,2020-04-16 01:15:47.540,True,Prime Video,NaN,4.8,1966.0,NaN,NaN,NaN
4,5,Way better than the harsh reviews.,Our family loved the film. We have kids and th...,B07RXM26FG,B07RXM26FG,2019-09-29 05:17:12.700,True,Prime Video,NaN,4.5,57962.0,NaN,NaN,NaN
5,5,Five Stars,Great movie! My kids are obsessed with HP!,B00271DNP4,B00271DNP4,2018-07-02 18:35:20.803,True,Prime Video,NaN,4.9,59229.0,NaN,NaN,NaN
6,5,Five Stars,LOVE this movie! Hits you in the feels!,B01LWY8995,B01LWY8995,2017-01-31 13:43:47.000,True,Prime Video,NaN,4.8,21628.0,NaN,NaN,NaN
7,5,Five Stars,LOVED Woodlawn & the story behind it!,B01AMUSHAC,B01AMUSHAC,2016-10-17 12:44:51.000,True,Prime Video,NaN,4.8,3522.0,NaN,NaN,NaN
8,5,Five Stars,Great Movie!,B00OI7J214,B00OI7J214,2015-11-03 19:40:37.000,True,Prime Video,NaN,4.6,16066.0,NaN,NaN,NaN
9,5,Five Stars,Great & exciting movie!,B016JGYSRY,B016JGYSRY,2015-10-27 17:57:20.000,True,Prime Video,NaN,4.6,23015.0,NaN,NaN,NaN


> **What we see:** The raw dataset has 14 columns including the review text (`text`), star rating (`rating`), timestamp, product metadata (title, description, price, categories) and user metadata (verified_purchase). Column names are not immediately intuitive — we will rename them for clarity.

---
## 1.2 Initial Inspection

Before touching the data, we need to fully understand what we are working with — data types, missing values, and duplicates.

In [5]:
# Data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 233610 entries, 0 to 233609
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             233610 non-null  int64  
 1   title_x            233586 non-null  object 
 2   text               233579 non-null  object 
 3   asin               233610 non-null  object 
 4   parent_asin        233610 non-null  object 
 5   timestamp          233610 non-null  object 
 6   verified_purchase  233610 non-null  bool   
 7   main_category      233610 non-null  object 
 8   title_y            33800 non-null   object 
 9   average_rating     233607 non-null  float64
 10  rating_number      233607 non-null  float64
 11  description        33800 non-null   object 
 12  price              24165 non-null   float64
 13  categories         33800 non-null   object 
dtypes: bool(1), float64(3), int64(1), object(9)
memory usage: 23.4+ MB


> **What we see:** Most columns have the correct data types. However, `timestamp` is an `object` (string) rather than `datetime` — we will fix this. The most striking observation is the massive number of null values in `title_y`, `description`, `price`, and `categories` — these come from the metadata dataset which only covered ~33K of the 233K products.

In [6]:
# Missing value analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print('Missing Values Summary:')
print('=' * 40)
print(missing_df[missing_df['Missing Count'] > 0].to_string())

Missing Values Summary:
                Missing Count  Missing %
price                  209445      89.66
title_y                199810      85.53
categories             199810      85.53
description            199810      85.53
title_x                    24       0.01
text                       31       0.01
average_rating              3       0.00
rating_number               3       0.00


> **What we see:** `description`, `categories`, `price`, and `title_y` (movie title) are missing in ~85% of rows. This is because the metadata dataset only covered a subset of the products. For our **modeling objective** (sentiment classification from review text), this is acceptable — we only need `text` and `rating`. However, for **EDA by genre**, we need `categories`, so we will create two datasets: a full dataset for modeling and a smaller genre-filtered one for EDA.

In [7]:
# Check for duplicate rows
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes:,} ({dupes/len(df)*100:.2f}% of dataset)')

Duplicate rows: 538 (0.23% of dataset)


> **What we see:** There are 538 duplicate rows — a very small fraction (~0.23%). These are likely cases where the same user submitted the same review multiple times. We will drop these to avoid any data leakage between train and test sets.

In [8]:
# Rating distribution before cleaning
print('Rating Distribution (raw):')
print('=' * 35)
rating_counts = df['rating'].value_counts().sort_index()
for rating, count in rating_counts.items():
    bar = '█' * int(count / 1000)
    print(f'  {rating} ⭐ : {count:>7,}  {bar}')

Rating Distribution (raw):
  1 ⭐ :  19,461  ███████████████████
  2 ⭐ :  13,953  █████████████
  3 ⭐ :  23,441  ███████████████████████
  4 ⭐ :  41,302  █████████████████████████████████████████
  5 ⭐ : 135,453  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████


> **What we see:** The dataset is heavily skewed toward 5-star ratings — a very common pattern in Amazon reviews known as **J-curve distribution**. This class imbalance means our models will have a natural bias toward predicting `positive`. We will address this during modeling with undersampling and class-weighted training.

---
## 1.3 Data Cleaning

We now clean the dataset in a clear, step-by-step manner. Each step is logged so we can track exactly how many rows we lose at each stage.

In [9]:
print('🧹 Starting data cleaning pipeline...')
print(f'   Starting shape: {df.shape}')
print()

# Step 1: Drop duplicate rows
df = df.drop_duplicates()
print(f'Step 1 — Drop duplicates      : {df.shape[0]:,} rows remaining')

# Step 2: Rename columns for clarity
df = df.rename(columns={
    'text'    : 'review_text',
    'title_x' : 'rating_desc',
    'title_y' : 'movie_title'
})
print(f'Step 2 — Rename columns       : done')

# Step 3: Drop columns not needed for modeling
df = df.drop(columns=['asin', 'parent_asin', 'rating_number', 'main_category'])
print(f'Step 3 — Drop unused columns  : {df.shape[1]} columns remaining')

# Step 4: Drop rows where review_text is null (can't model without text)
before = len(df)
df = df.dropna(subset=['review_text'])
print(f'Step 4 — Drop null review_text: {len(df):,} rows remaining (lost {before - len(df):,})')

# Step 5: Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
print(f'Step 5 — Parse timestamps     : done')

# Step 6: Drop rows with invalid timestamps
before = len(df)
df = df.dropna(subset=['timestamp'])
print(f'Step 6 — Drop bad timestamps  : {len(df):,} rows remaining (lost {before - len(df):,})')

# Step 7: Reset index
df = df.reset_index(drop=True)
print(f'Step 7 — Reset index          : done')
print()
print(f'✅ Cleaning complete. Final shape: {df.shape}')

🧹 Starting data cleaning pipeline...
   Starting shape: (233610, 14)

Step 1 — Drop duplicates      : 233,072 rows remaining
Step 2 — Rename columns       : done
Step 3 — Drop unused columns  : 10 columns remaining
Step 4 — Drop null review_text: 233,041 rows remaining (lost 31)
Step 5 — Parse timestamps     : done
Step 6 — Drop bad timestamps  : 233,041 rows remaining (lost 0)
Step 7 — Reset index          : done

✅ Cleaning complete. Final shape: (233041, 10)


> **What we see:** We lost very few rows during cleaning — the dataset is mostly intact. The primary step was dropping duplicate rows and null review texts. We now have a clean, well-typed DataFrame ready for feature engineering.

---
## 1.4 Feature Engineering

We now build the key features the project depends on:
1. A **`sentiment`** label column derived from star ratings
2. A combined **`reviews`** column that joins `rating_desc` + `review_text` for richer text context
3. Extracted **`year`** and **`month`** from the timestamp for time-based EDA

In [10]:
# Assign sentiment labels using our src/preprocess.py function
df['sentiment'] = df['rating'].apply(assign_sentiment)

# Verify the mapping
print('Sentiment Label Distribution:')
print('=' * 40)
sentiment_counts = df['sentiment'].value_counts()
for label, count in sentiment_counts.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {label:<10}: {count:>7,}  ({pct:.1f}%)  {bar}')

Sentiment Label Distribution:
  positive  : 176,303  (75.7%)  █████████████████████████████████████
  negative  :  33,343  (14.3%)  ███████
  neutral   :  23,395  (10.0%)  █████


> **What we see:** The class imbalance is stark — `positive` reviews dominate at ~77%, while `neutral` and `negative` are a small minority. This reflects real-world review behavior: people who love something are more likely to leave a review than those with mixed feelings. This imbalance is a major modeling challenge we will address in Notebook 03.

In [12]:
# Combine rating_desc and review_text into a single 'reviews' column
# rating_desc adds useful context (e.g. 'Five Stars', 'Disappointed') to the raw review text
df['reviews'] = (
    df['rating_desc'].fillna('') + '  ' + df['review_text'].fillna('')
).str.strip()

# Extract year and month for time-based analysis in EDA
df['year']  = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month

print('✅ Feature engineering complete.')
print(f"   New columns added: 'sentiment', 'reviews', 'year', 'month'")
df[['rating', 'rating_desc', 'review_text', 'reviews', 'sentiment']].head(5)

✅ Feature engineering complete.
   New columns added: 'sentiment', 'reviews', 'year', 'month'


,rating,rating_desc,review_text,reviews,sentiment
0,5,Five Stars,"Amazon, please buy the show! I'm hooked!","Five Stars Amazon, please buy the show! I'm h...",positive
1,5,Five Stars,My Kiddos LOVE this show!!,Five Stars My Kiddos LOVE this show!!,positive
2,5,What Love Is...,"...isn't always how you expect it to be, but w...",What Love Is... ...isn't always how you expec...,positive
3,5,QUIRKY TURNS TO HEARTSTRINGS,As you learn about the very unique characters ...,QUIRKY TURNS TO HEARTSTRINGS As you learn abo...,positive
4,5,Way better than the harsh reviews.,Our family loved the film. We have kids and th...,Way better than the harsh reviews. Our family...,positive


> **What we see:** The `reviews` column now contains the full context of each review. For example, a review with `rating_desc='Five Stars'` and `review_text='My kiddos love this show!'` becomes `'Five Stars  My kiddos love this show!'` — giving the model a stronger signal.

---
## 1.5 Build Final Datasets

We create **two versions** of the clean dataset:

- **Full dataset** (`233K rows`) — used for modeling. Contains all reviews with `review_text`, `sentiment`, and timestamps.
- **Genre dataset** (`~33K rows`) — subset where `categories` is not null. Used for genre-based EDA in Notebook 02.

In [13]:
# ── Full modeling dataset ─────────────────────────────────────
model_cols = ['rating', 'timestamp', 'year', 'month',
              'review_text', 'reviews', 'sentiment']
df_model = df[model_cols].copy()

# ── Genre EDA dataset ─────────────────────────────────────────
# Keep only rows with metadata, clean up categories column
df_genre = df.dropna(subset=['categories', 'movie_title']).copy()
df_genre = df_genre[~df_genre['categories'].str.contains('[', regex=False)]
df_genre['price'] = df_genre['price'].fillna(df_genre['price'].mean())
df_genre = df_genre.dropna(subset=['review_text'])

genre_cols = ['rating', 'timestamp', 'year', 'month', 'movie_title',
              'price', 'categories', 'review_text', 'reviews', 'sentiment']
df_genre = df_genre[genre_cols].reset_index(drop=True)

print('📊 Final Dataset Summary:')
print('=' * 45)
print(f'  Full modeling dataset : {df_model.shape[0]:>7,} rows × {df_model.shape[1]} cols')
print(f'  Genre EDA dataset     : {df_genre.shape[0]:>7,} rows × {df_genre.shape[1]} cols')

📊 Final Dataset Summary:
  Full modeling dataset : 233,041 rows × 7 cols
  Genre EDA dataset     :  33,554 rows × 10 cols


In [14]:
# Final validation — make sure there are no nulls in critical columns
print('🔍 Validation — Null Check on Critical Columns:')
print('=' * 50)

for col in ['review_text', 'reviews', 'sentiment', 'rating']:
    nulls = df_model[col].isnull().sum()
    status = '✅' if nulls == 0 else '❌'
    print(f'  {status} df_model["{col}"] — {nulls} nulls')

print()
# Confirm all sentiment labels are valid
valid_labels = set(CONFIG['sentiment']['labels'])
invalid = df_model[~df_model['sentiment'].isin(valid_labels)]
print(f'  ✅ All sentiment labels valid: {df_model["sentiment"].unique().tolist()}')
print(f'  ✅ No invalid labels found: {len(invalid)} rows')

🔍 Validation — Null Check on Critical Columns:
  ✅ df_model["review_text"] — 0 nulls
  ✅ df_model["reviews"] — 0 nulls
  ✅ df_model["sentiment"] — 0 nulls
  ✅ df_model["rating"] — 0 nulls

  ✅ All sentiment labels valid: ['positive', 'neutral', 'negative']
  ✅ No invalid labels found: 0 rows


---
## 1.6 Save Processed Datasets

In [15]:
PROCESSED_PATH = '../data/processed/'
os.makedirs(PROCESSED_PATH, exist_ok=True)

# Save both datasets
df_model.to_csv(f'{PROCESSED_PATH}reviews_model.csv', index=False)
df_genre.to_csv(f'{PROCESSED_PATH}reviews_genre.csv', index=False)

print('💾 Datasets saved:')
print(f'   → data/processed/reviews_model.csv  ({df_model.shape[0]:,} rows)')
print(f'   → data/processed/reviews_genre.csv  ({df_genre.shape[0]:,} rows)')

💾 Datasets saved:
   → data/processed/reviews_model.csv  (233,041 rows)
   → data/processed/reviews_genre.csv  (33,554 rows)


---
## 1.7 Summary

Here is a recap of everything accomplished in this notebook:

| Step | Action | Result |
|---|---|---|
| Load | Read raw `prime_videos.csv` | 233,610 rows × 14 cols |
| Inspect | Checked dtypes, nulls, duplicates | Found 538 dupes, ~85% nulls in metadata |
| Clean | Dropped dupes, nulls, renamed cols | ~233,500 clean rows |
| Engineer | Added `sentiment`, `reviews`, `year`, `month` | 4 new features |
| Split | Full dataset vs genre subset | 233K modeling / 33K EDA |
| Save | Saved to `data/processed/` | 2 clean CSV files |

### ⚠️ Key Observations to Carry Forward

1. **Class Imbalance** — ~77% positive, ~11% neutral, ~12% negative. Must be addressed in modeling.
2. **Rich metadata is sparse** — only ~14% of reviews have genre/title data. Genre EDA is limited to this subset.
3. **Combined `reviews` column** — joining `rating_desc` + `review_text` gives richer text signal than review text alone.

---

**Next:** `02_eda.ipynb` — Exploratory Data Analysis & Visualization